# ⚡ Caching — Complete Guide

> w Definitier:**
> *"A cache is a fast, temporary storage layer that sits between your application and your slow data source. Instead of fetching data from the database every time, you store a copy closer to the application and serve it from there — trading memory for speed."*

---

## Table of Contents

1. [Why Cache is Important](#1-why-cache-is-important)
2. [Cache Hit and Cache Miss](#2-cache-hit-and-cache-miss)
3. [TTL — Time To Live](#3-ttl--time-to-live)
4. [Cache Write Strategies](#4-cache-write-strategies)
   - [Read Through Cache](#41-read-through-cache)
   - [Write Through Cache](#42-write-through-cache)
   - [Write Around Cache](#43-write-around-cache)
   - [Write Back Cache](#44-write-back-cache)
5. [Cache Eviction Policies](#5-cache-eviction-policies)
   - [LRU — Least Recently Used](#51-lru--least-recently-used)
   - [LFU — Least Frequently Used](#52-lfu--least-frequently-used)
   - [FIFO — First In First Out](#53-fifo--first-in-first-out)
   - [MRU — Most Recently Used](#54-mru--most-recently-used)
   - [LIFO — Last In First Out](#55-lifo--last-in-first-out)
6. [Eviction Policies Comparison](#6-eviction-policies-comparison)
7. [Quick Revision Cheatsheet](#7-quick-revision-cheatsheet)

---

## 1. Why Cache is Important

### 🧠 Simple Definition
Caching is storing the **result of an expensive operation** in fast memory so the next request for the same thing is served instantly — without repeating the expensive work.

> **Interview Line to Remember:**
> *"Cache exists because memory is fast and disk/network is slow. If the same data is requested repeatedly, computing or fetching it every time wastes time and resources. Cache trades cheap memory for expensive CPU and DB round trips."*

---

### ❓ What Problem Does It Solve?

Three fundamental problems:

**1. Latency** — A database query takes 50–200ms. A cache read takes < 1ms. For high-traffic APIs, that difference compounds into seconds of user-perceived slowness.

**2. Database Overload** — 10,000 users requesting the same product page = 10,000 identical DB queries hitting the same rows. Cache collapses that to one DB query + 9,999 cache reads.

**3. Computation Cost** — Aggregations, reports, ML recommendations take seconds to compute. Cache the result — serve it in microseconds for every subsequent request.

### 🏭 Why Needed in Production?
- Reduces **p99 latency** — the slowest 1% of requests that drag down user experience
- Protects DB from **thundering herd** — sudden traffic spike doesn't cascade into DB meltdown
- Enables **horizontal scaling** — stateless app servers all read the same shared cache (Redis)
- Cuts **infrastructure cost** — fewer DB read replicas needed when cache absorbs 80%+ of reads
- Powers **real-time features** — leaderboards, live counters, session management at millisecond speed

### 💥 What Happens Without It?
- Every page load hits the database → DB CPU spikes to 100% → queries queue → timeouts → site down
- A viral product on Flipkart with no cache: 1M users refresh the page → 1M identical DB queries → server crash
- Complex recommendation API (Netflix): computed fresh per request → 2-second response time → user leaves
- Auth check on every API call hits DB → bottleneck at 10k req/sec → login system collapses

### 🏢 Real Company Example
**Twitter/X** — Your home timeline isn't computed live from the database. It's **pre-computed and stored in Redis**. When you open Twitter, you're reading from a cache. Without caching, serving 500M daily users' timelines from PostgreSQL would be physically impossible.

---

### 📐 Where Cache Lives in the Stack

```
User Request
     │
     ▼
[Application Server]
     │
     ▼ Check cache first
[Cache Layer - Redis]  ──── HIT → Return in <1ms ──────────────► Response
     │
     │ MISS
     ▼
[Database - PostgreSQL]  ──── fetch data (50-200ms)
     │
     ▼
Store result in cache → Return to user
```

---

### 📐 Speed Comparison — Why Cache Wins

| Storage Layer     | Approximate Latency   | Relative Speed      |
|------------------|-----------------------|---------------------|
| CPU L1 Cache     | ~0.5 ns               | Fastest possible    |
| RAM (memory)     | ~100 ns               | Very fast           |
| **Redis Cache**  | **~0.1 – 1 ms**       | **Fast**            |
| SSD Disk         | ~0.1 ms               | Moderate            |
| Database Query   | ~10 – 200 ms          | Slow                |
| Network API call | ~50 – 500 ms          | Very slow           |

> **The point:** A cache read is **100–1000x faster** than a database query. At 10,000 requests/second, this difference is the line between "site works" and "site is down."

---

## 2. Cache Hit and Cache Miss

### 🧠 Simple Definition
- **Cache Hit** — The requested data IS in the cache. Served immediately. Fast, cheap, perfect.
- **Cache Miss** — The requested data is NOT in the cache. Must fetch from the original source (DB/API), then optionally store it in cache for next time.

> **Interview Line to Remember:**
> *"Cache Hit = data found in cache → serve instantly. Cache Miss = data not in cache → go to DB, fetch it, store it in cache, then serve. Hit rate is the key metric — a good cache should have 90%+ hit rate in production."*

---

### ❓ What Problem Does It Solve?
Tracking hits vs misses tells you **how effective your cache is**. A low hit rate means cache isn't helping — you're paying the cost of the cache layer without the benefit. It drives decisions about what to cache, how long to cache it, and how much memory to allocate.

### 🏭 Why Needed in Production?
- **Cache Hit Rate** is a primary SRE/DevOps metric — monitored on Datadog, Grafana
- Hit rate dropping suddenly = cache server died or memory filled up = DB about to get hammered
- Helps size the cache — if hit rate is 60%, you need more memory or a better eviction policy
- Guides caching strategy — what data to pre-warm, what TTLs to set

### 💥 What Happens Without Tracking It?
You won't know your cache is doing nothing. Engineers assume "we have Redis" = fast system, but if hit rate is 20%, you have a slow system with an extra Redis bill.

### 🏢 Real Company Example
**Cloudflare CDN** — their dashboard shows cache hit rate per domain. A well-configured site has 85–95% hit rate. Static assets (images, JS) always hit. Dynamic personalized pages always miss. Engineers optimize what they cache based on this metric.

---

### 📐 Cache Hit vs Miss Flow

```
Request for user_id = 42 profile

Cache HIT:
────────────────────────────────
App → Redis.GET("user:42") → "Alice, alice@.." ✅
Response time: 0.5ms
DB: not touched

Cache MISS:
────────────────────────────────
App → Redis.GET("user:999") → nil ❌
App → DB.query("SELECT * FROM users WHERE id=999") → {name: Bob...}
App → Redis.SET("user:999", "{name: Bob...}", EX 3600)
App → Return data to user
Response time: 120ms
DB: queried once, next 3600 seconds will be cache hits
```

### 📐 Hit Rate Formula

```
Cache Hit Rate = (Cache Hits) / (Cache Hits + Cache Misses) × 100

Example:
  1000 requests total
  850 cache hits
  150 cache misses

  Hit Rate = 850 / 1000 × 100 = 85%   ← Good
  Miss Rate = 15%

Target in production: > 90% for frequently-accessed data
```

---

## 3. TTL — Time To Live

### 🧠 Simple Definition
TTL (Time To Live) is a **timer set on a cache entry** that automatically expires and deletes it after a specified duration. Once expired, the next request triggers a fresh fetch from the source.

> **Interview Line to Remember:**
> *"TTL is the expiry date on cached data. It answers: how long can I serve this data before it might be stale? Short TTL = fresher data, more DB hits. Long TTL = more cache hits, risk of serving stale data. TTL is the tuning dial between freshness and performance."*

---

### ❓ What Problem Does It Solve?
Cached data can become **stale** — the DB has updated but the cache still holds old values. Without TTL, you'd serve wrong data forever (or until a manual cache flush). TTL automates expiry so stale data self-destructs after a defined period.

### 🏭 Why Needed in Production?
- **Prevents stale data** — prices, inventory, user balances must not be served from stale cache
- **Automatic memory management** — expired keys are freed, cache doesn't fill up with dead data
- **Compliance** — GDPR: deleted user data can't persist in cache after DB deletion. TTL ensures it expires
- **Different TTLs for different data** — stock prices: 5 seconds. User profiles: 1 hour. Static content: 7 days

### 💥 What Happens Without It?
- Cache entry never expires → user sees wrong price/balance/status forever
- Memory fills with data nobody needs anymore → cache evicts hot data → hit rate collapses
- User deletes account → their data remains in cache → privacy violation

### 🏢 Real Company Example
**Zerodha (stock trading):**
- Live stock price → TTL: **5 seconds** (must be near-real-time)
- User portfolio summary → TTL: **60 seconds** (acceptable slight delay)
- Historical chart data → TTL: **1 hour** (doesn't change)
- Static help articles → TTL: **24 hours** (rarely changes)

---

### 📐 TTL in Practice (Redis)

```bash
# Set with TTL
SET  user:42:profile  '{"name":"Alice"}'  EX 3600    # Expires in 1 hour
SET  stock:RELIANCE   '4250.75'           EX 5        # Expires in 5 seconds
SET  homepage:html    '<html>...'         EX 86400    # Expires in 24 hours

# Check remaining TTL
TTL user:42:profile   → 3547  (seconds remaining)
TTL stock:RELIANCE    → 2     (seconds remaining)

# Key expired — next GET triggers DB fetch
GET stock:RELIANCE    → nil   (TTL elapsed, key gone)
```

### 📐 TTL Strategy by Data Type

| Data Type                  | Recommended TTL  | Why                                          |
|---------------------------|------------------|----------------------------------------------|
| Live prices / rates        | 1 – 10 seconds   | Changes every second                         |
| User session tokens        | 30 – 60 minutes  | Security + convenience balance               |
| User profile data          | 1 – 6 hours      | Changes rarely, personalization needed       |
| Product catalog / listings | 6 – 24 hours     | Updated infrequently                         |
| Static content / configs   | 1 – 7 days       | Rarely changes                               |
| Computed reports / analytics | 1 – 6 hours    | Expensive to recompute, slight staleness OK  |

---

## 4. Cache Write Strategies

> **Interview Overview:**
> *"There are four ways to keep cache and database in sync. The strategy you pick determines: who populates the cache, how fresh the data is, and what happens if the cache goes down. Each trades off between consistency, write speed, and complexity."*

---

### 4.1 Read Through Cache

### 🧠 Simple Definition
In Read Through, the **cache sits in front of the database**. The application always talks to the cache — never directly to the DB. On a cache miss, the **cache itself** fetches the data from the DB, stores it, and returns it.

> **Interview Line to Remember:**
> *"Read Through: app talks only to cache. Cache miss? Cache fetches from DB itself, stores it, returns it. App never touches DB directly. Cache is the single source of truth for reads."*

---

### ❓ What Problem Does It Solve?
Lazy population — cache only stores data that's actually requested. You don't pre-load everything — the cache fills itself organically as real requests come in.

### 🏭 Why Needed in Production?
- Application code stays simple — always read from cache, never conditionally check DB
- Cache auto-populates on first miss — no manual cache warming needed
- Works well with CDNs and managed cache services (AWS ElastiCache)

### 💥 What Happens Without It?
Application code gets messy — every read must check cache, handle miss, query DB, write to cache, return. Logic scattered. Read Through centralizes this in the cache layer.

### 🏢 Real Company Example
**AWS ElastiCache with RDS** — most e-commerce platforms configure ElastiCache in read-through mode. When a product page loads, ElastiCache checks its memory. If not found, it fetches from RDS Aurora and caches it automatically.

---

### 📐 Read Through Flow

```
Read Through:

First Request (Cache Miss):
  App ──► Cache ── MISS ──► DB ──► returns data
                    │               │
                    └── stores ◄────┘
                    │
  App ◄────────────┘ (returns data to app)

Second Request (Cache Hit):
  App ──► Cache ── HIT ──► returns data instantly
  DB:  never touched

Key Characteristic: Cache is the only layer app talks to.
```

---

### 4.2 Write Through Cache

### 🧠 Simple Definition
In Write Through, **every write goes to the cache AND the database simultaneously** (or cache first, then DB synchronously). Data is always consistent between cache and DB — but every write pays the cost of writing to both.

> **Interview Line to Remember:**
> *"Write Through: every write updates cache AND database at the same time. Cache and DB are always in sync — no stale data. But writes are slower because both writes must complete before confirming success."*

---

### ❓ What Problem Does It Solve?
The classic cache consistency problem: if you update the DB but not the cache, reads from cache return stale data. Write Through ensures **cache never serves stale data** because it's updated on every write.

### 🏭 Why Needed in Production?
- Financial data — account balance updated in DB must immediately be in cache
- User profile — just-updated profile should be served fresh from cache instantly
- Inventory — product stock level must be consistent between cache and DB

### 💥 What Happens Without It?
Write only to DB, forget to update cache → user updates their address → still sees old address for TTL duration → support tickets, bad user experience.

### 🏢 Real Company Example
**Paytm wallet balance** — when you add money or pay someone, the new balance is written to both the DB and Redis cache simultaneously. Next time you open the app, you see the correct balance instantly from cache — never a stale figure.

---

### 📐 Write Through Flow

```
Write Through:

Write Request (update user email):
  App ──► Cache.SET("user:42", newData) ──► DB.UPDATE users SET email=...
           │                                    │
           └─────── both complete ──────────────┘
           │
  App ◄─── Success confirmed

Next Read:
  App ──► Cache ── HIT ──► returns FRESH updated data ✅

Pros: Always consistent. No stale reads.
Cons: Write latency = cache write time + DB write time (both must succeed).
```

---

### 4.3 Write Around Cache

### 🧠 Simple Definition
In Write Around, **writes go directly to the database, bypassing the cache entirely**. The cache is only populated on a read miss. New data isn't in cache until someone requests it.

> **Interview Line to Remember:**
> *"Write Around: writes skip the cache and go straight to DB. Cache is populated only on read miss. Best for write-heavy data that's rarely read — no point filling cache with data nobody will request."*

---

### ❓ What Problem Does It Solve?
Write Through wastes cache memory on data that's written but never read again (logs, audit trails, bulk imports). Write Around avoids polluting cache with data that won't be re-read.

### 🏭 Why Needed in Production?
- Log ingestion — billions of log entries written, almost never individually re-read
- Batch data imports — loading 1M product records doesn't mean all 1M will be immediately read
- Reporting inserts — writing aggregated data to DB for BI tools that read directly from DB

### 💥 What Happens Without It?
Write Through on write-heavy workloads → cache fills with data that's read once or never → evicts actually-hot data → hit rate collapses → performance degrades.

### 🏢 Real Company Example
**Datadog / Splunk log ingestion** — millions of log events per second are written directly to storage. No caching on write. If an engineer queries a specific log line, it's fetched from storage and cached then. Caching every log write would be memory suicide.

---

### 📐 Write Around Flow

```
Write Around:

Write Request:
  App ──► DB directly (cache bypassed entirely)
  Cache:  not touched on write ❌

Read Request (first time after write):
  App ──► Cache ── MISS ──► DB ──► data returned
                    │
                    └── stored in cache now

Read Request (second time):
  App ──► Cache ── HIT ── data served instantly ✅

Pros:  Cache not polluted with write-heavy data. Good memory efficiency.
Cons:  First read after write always misses (cold read penalty).
```

---

### 4.4 Write Back Cache (Write Behind)

### 🧠 Simple Definition
In Write Back (also called Write Behind), **writes go to the cache first and are immediately confirmed as successful**. The cache then **asynchronously writes to the database in the background** — batching multiple writes into fewer DB operations.

> **Interview Line to Remember:**
> *"Write Back: write to cache first, confirm immediately to the user. Cache writes to DB asynchronously later. Fastest writes possible. Risk: if cache dies before DB is updated, that data is lost. High performance, lower durability."*

---

### ❓ What Problem Does It Solve?
High-write-throughput scenarios where write latency must be minimal. Instead of paying the DB write cost on every operation, batch many writes and flush to DB periodically — one DB write for 100 cache writes.

### 🏭 Why Needed in Production?
- Real-time counters — "views: 1,000,000" incremented millions of times, DB updated every 10 seconds
- Like/heart counts on social media — too expensive to DB-write every like instantly
- Gaming leaderboards — scores updated constantly, DB synced every few seconds
- IoT sensor data — thousands of readings per second, DB can't handle the write rate

### 💥 What Happens Without It?
Direct DB writes for every like/view/click → DB saturated → writes queue → latency spikes → app slow. Write Back absorbs the burst and smooths it out.

### ⚠️ The Risk:
Cache crashes between write confirmation and DB flush → **data loss**. Never use for financial transactions. Acceptable for metrics, counters, non-critical state.

### 🏢 Real Company Example
**YouTube view counter** — "1 Billion views" is tracked in Redis. Every play increments a Redis counter (sub-millisecond). Redis periodically flushes the batch to MySQL. If Redis crashes and loses 10,000 pending view counts — YouTube doesn't care. If a bank lost 10,000 transactions — disaster.

---

### 📐 Write Back Flow

```
Write Back (Write Behind):

Write Request (100 users click "Like" simultaneously):
  User 1-100 ──► Cache.INCR("post:likes") × 100   ← instant, in memory
  Response:       "Liked!" ✅  confirmed immediately
  DB:             not touched yet

Background async worker (every 10 seconds):
  Cache ──► DB: UPDATE posts SET likes = 100 WHERE id = X

Next Read:
  App ──► Cache ── HIT ──► returns 100 likes ✅

Pros:  Extremely fast writes. DB not hammered on every write.
Cons:  Data loss risk if cache dies before async flush to DB.
       Complexity of async reconciliation.
```

---

### 📐 Write Strategy Comparison Table

| Strategy          | Write goes to      | Cache populated     | Read consistency | Write speed    | Data loss risk   |
|------------------|--------------------|---------------------|-----------------|----------------|-----------------|
| **Read Through** | DB directly        | On cache miss       | May be stale    | Normal         | None            |
| **Write Through**| Cache + DB (sync)  | On every write      | Always fresh    | Slower (2 writes) | None         |
| **Write Around** | DB directly        | On read miss only   | May be stale    | Normal         | None            |
| **Write Back**   | Cache first        | On every write      | Fresh from cache| Fastest        | ⚠️ Yes (if crash)|

---

## 5. Cache Eviction Policies

### 🧠 Simple Definition
Cache has **finite memory**. When it's full and a new item needs to be stored, something old must be removed. An **Eviction Policy** is the algorithm that decides **which item to kick out** to make room.

> **Interview Line to Remember:**
> *"Eviction policy is the bouncer at the door — when cache is full and a new item arrives, the bouncer decides which existing item gets thrown out. The goal is always to evict data least likely to be needed again soon."*

---

### ❓ What Problem Does It Solve?
Memory is finite. Without a strategy, either the cache fills up and rejects new items (useless) or you evict randomly (inefficient). A good eviction policy maximizes cache hit rate within a fixed memory budget.

### 🏭 Why Needed in Production?
- Redis has a `maxmemory` limit. When reached, it uses the configured eviction policy
- Wrong policy → evict hot data → hit rate drops → DB gets hammered
- Right policy for workload → hot data stays, cold data leaves → hit rate stays high
- Different parts of a system may need different policies

---

### 5.1 LRU — Least Recently Used

### 🧠 Simple Definition
LRU evicts the item that was **accessed the longest time ago** — the item nobody has touched the most recently. The assumption: if you haven't needed it recently, you probably won't need it soon.

> **Interview Line to Remember:**
> *"LRU evicts the item not touched for the longest time. Assumption: recently used = likely to be used again. The most popular eviction policy for general caching — Redis uses it by default."*

---

### ❓ What Problem Does It Solve?
General temporal locality — recently accessed data tends to be accessed again soon. Items not accessed in a long time are likely "cold" and safe to evict.

### 🏭 Why Needed in Production?
- Web caches — recently visited pages are more likely to be revisited
- Session caches — active sessions are recently touched, idle ones are old
- API response caches — popular endpoints hit repeatedly in the same time window
- **Redis default** — `maxmemory-policy allkeys-lru`

### 💥 What Happens Without It?
Without LRU, you might evict a product page viewed 10 seconds ago to make room for one viewed once a year ago — terrible cache efficiency.

### 🏢 Real Company Example
**Netflix browse page** — recently viewed titles are LRU-cached. If you viewed Stranger Things 2 seconds ago, it stays in cache. A documentary you watched 3 months ago gets evicted first.

---

### 📐 LRU Visualization

```
Cache Size: 4 slots
Access order: A, B, C, D, E (E is new, cache is full)

State before E arrives:
  [D(newest)] [C] [B] [A(oldest)]

Evict: A (Least Recently Used)
Insert: E

State after:
  [E(newest)] [D] [C] [B]
   A is gone
```

**LRU Data Structure (Interview Bonus):**
LRU is implemented with a **HashMap + Doubly Linked List**:
- HashMap: O(1) lookup by key
- Linked List: O(1) move to front on access, O(1) remove from tail on eviction

---

### 5.2 LFU — Least Frequently Used

### 🧠 Simple Definition
LFU evicts the item that has been **accessed the fewest number of times** overall. It tracks a frequency counter for every item — the item with the lowest count gets evicted first.

> **Interview Line to Remember:**
> *"LFU evicts the item accessed fewest total times — it counts accesses, not recency. A popular item accessed 1000 times stays over a one-time item, even if the popular one hasn't been touched in a while."*

---

### ❓ What Problem Does It Solve?
LRU's weakness: a popular item accessed 10,000 times gets evicted if it wasn't touched in the last minute. LFU solves this by counting total access frequency — truly popular items stay cached regardless of recency.

### 🏭 Why Needed in Production?
- CDN/content caches — viral videos (accessed millions of times) should never be evicted
- Product catalog — bestselling products should stay cached even during brief lulls in traffic
- API response caches — popular endpoints with stable responses

### 💥 What Happens Without It?
With LRU, a brief traffic lull can cause your most popular item to be evicted. If nobody requests iPhone 15 for 5 minutes (everyone's at lunch), LRU might evict it — then everyone comes back and causes a cache miss tsunami.

### 🏢 Real Company Example
**Spotify song cache** — "Blinding Lights" has been streamed 4 billion times. Even if nobody streamed it in the last 5 minutes, LFU keeps it cached because its frequency count is astronomically high compared to a deep-cut B-side.

---

### 📐 LRU vs LFU — The Classic Contrast

```
Cache: 3 slots
Items and access counts: A=100x, B=1x, C=50x

New item D arrives → must evict something

LRU Decision: Evict whoever was least RECENTLY accessed
  → If B was accessed 10 seconds ago, A 5 seconds ago, C 2 seconds ago
  → Evict A (oldest access) even though A was accessed 100 times ❌ (wrong!)

LFU Decision: Evict whoever was accessed FEWEST times
  → B accessed only 1 time
  → Evict B ✅ (correct — B is the cold item)
```

---

### 5.3 FIFO — First In First Out

### 🧠 Simple Definition
FIFO evicts the item that was **inserted into the cache first**, regardless of how often or recently it was accessed. First in, first out — like a queue.

> **Interview Line to Remember:**
> *"FIFO is a queue — the oldest inserted item is evicted first, regardless of how popular or recently used it is. Simple to implement. Not smart — ignores usage patterns entirely."*

---

### ❓ What Problem Does It Solve?
Simplicity. When you need a basic eviction strategy with minimal overhead and your data access pattern is uniformly distributed (all items equally likely to be accessed), FIFO works and has O(1) eviction.

### 🏭 Why Needed in Production?
- Message queues and job queues where order matters
- Streaming data pipelines — process and evict in arrival order
- Log buffers — oldest log entries flushed to disk first
- Simple embedded systems with constrained processing (IoT devices)

### 💥 What Happens Without It?
FIFO's weakness: evicts a highly popular item simply because it was inserted first. "Google.com" homepage cached 10,000 requests ago gets evicted before a one-time request cached 10 seconds ago. Wrong priority.

### 🏢 Real Company Example
**Print spoolers, OS disk buffers, message brokers (Kafka)** — data is written and consumed in order. The "cache" (buffer) evicts the oldest written data first. Order matters more than frequency here.

---

### 📐 FIFO Visualization

```
Cache Size: 4 slots (insertion order tracked)
Insert order: A(t=1), B(t=2), C(t=3), D(t=4)

New item E arrives (t=5):
  Queue: [A(t=1)] [B(t=2)] [C(t=3)] [D(t=4)]

FIFO evicts: A (oldest insert time)

State after:
  [B(t=2)] [C(t=3)] [D(t=4)] [E(t=5)]

Note: Even if A was accessed 1000 times, FIFO still evicts it — blindly.
```

---

### 5.4 MRU — Most Recently Used

### 🧠 Simple Definition
MRU evicts the item that was **accessed most recently** — the exact opposite of LRU. The assumption: if you just used it, you probably won't need it again immediately.

> **Interview Line to Remember:**
> *"MRU evicts the MOST recently used item — opposite of LRU. Sounds counterintuitive, but it's correct for sequential scan workloads where you access each item once and never again. The just-accessed item is the safest to evict."*

---

### ❓ What Problem Does It Solve?
Sequential/one-time-scan access patterns — if you're reading through a massive dataset sequentially and each item is accessed exactly once, the most recently accessed item is the least likely to be needed again. LRU would keep it; MRU correctly evicts it.

### 🏭 Why Needed in Production?
- **Database buffer pools during full table scans** — scanning a huge table once; each page is read once; evict the page just read to make room for the next one
- **Backup and archival processes** — reading files sequentially to back them up
- **Report generation** — scanning historical data once for a report; just-scanned data is done

### 💥 What Happens Without It?
Using LRU on a sequential scan fills your cache entirely with the last-scanned items of the scan, evicting everything else — useless. MRU is the correct strategy here.

### 🏢 Real Company Example
**PostgreSQL's buffer cache** uses MRU-like behavior during sequential scans. When scanning a 100GB table for a report, PostgreSQL uses a "ring buffer" (MRU variant) so the massive scan doesn't pollute the buffer cache and evict the hot index pages other queries need.

---

### 📐 MRU Visualization

```
Sequential scan — reading items A, B, C, D, E in order, never revisiting

Cache Size: 3 slots
After reading A, B, C: [C(newest)] [B] [A]

Read D — cache full, must evict:
MRU evicts: C (Most Recently Used) ✅ because we just accessed it and won't again
Insert D: [D(newest)] [B] [A]

Read E — cache full:
MRU evicts: D (just accessed)
Insert E: [E(newest)] [B] [A]

Result: Cache never fills with stale scan data.
LRU would have evicted A first — but we might still read further items
similar to A's position, so keeping A makes no sense for sequential scan.
```

---

### 5.5 LIFO — Last In First Out

### 🧠 Simple Definition
LIFO evicts the item that was **inserted most recently** — the newest item leaves first when space is needed. Like a stack — last item pushed is the first popped.

> **Interview Line to Remember:**
> *"LIFO is a stack — the most recently inserted item is evicted first. Rare in caching. Useful when newer data is less valuable than established older data — undo stacks, call stacks, backtracking algorithms."*

---

### ❓ What Problem Does It Solve?
Niche scenarios where newer insertions are less valuable than established cached items. A stable, long-running cache of hot items should not be disturbed by new items that haven't yet proven their worth.

### 🏭 Why Needed in Production?
- **Undo/redo stacks** in text editors — most recent action is undone first
- **Browser history stack** — back button pops the most recent page
- **Backtracking in algorithms** — DFS, game trees, puzzle solvers
- Rarely the best policy for pure read caching — mainly useful in stack-oriented computation

### 💥 What Happens Without It?
LIFO would be terrible for a web cache — your most recently and likely-to-be-requested-again items would be evicted first. It's the right tool only in stack-based computation contexts.

### 🏢 Real Company Example
**Code editors (VS Code, IntelliJ)** — the undo history is a LIFO stack. Your most recent edit is the first one undone when you hit Ctrl+Z. This is the most natural and expected behavior for editing tools.

---

## 6. Eviction Policies Comparison

### 📐 Side-by-Side Summary

| Policy   | Evicts                    | Data Structure          | Time Complexity | Best For                                  | Weakness                                  |
|----------|--------------------------|-------------------------|-----------------|-------------------------------------------|-------------------------------------------|
| **LRU**  | Least recently accessed   | HashMap + Doubly LinkedList | O(1)        | General web caching, sessions             | Doesn't account for frequency             |
| **LFU**  | Least frequently accessed | HashMap + Min-Heap      | O(log n)        | Viral content, stable popular items       | Cold start — new popular items start at 0 |
| **FIFO** | Oldest inserted           | Queue                   | O(1)            | Simple buffers, message queues            | Ignores usage — evicts popular old items  |
| **MRU**  | Most recently accessed    | HashMap + Doubly LinkedList | O(1)        | Sequential scans, one-time read patterns  | Wrong for most web caching scenarios      |
| **LIFO** | Most recently inserted    | Stack                   | O(1)            | Undo stacks, DFS, backtracking            | Terrible for read caches                  |

---

### 📐 Which Policy for Which Scenario

```
General web caching (most use cases)?
    └─ LRU — safe default, Redis uses this

Viral/popular content that must stay cached?
    └─ LFU — frequency beats recency

Sequential scan / one-time read of large dataset?
    └─ MRU — just-read item is done, evict it

Simple ordered buffer / message queue?
    └─ FIFO — oldest processed first

Undo history / algorithm backtracking?
    └─ LIFO — newest action undone first

Redis maxmemory-policy config options:
    allkeys-lru      ← most common for general caching
    allkeys-lfu      ← for frequency-based workloads
    volatile-lru     ← LRU only among keys with TTL set
    volatile-lfu     ← LFU only among keys with TTL set
    allkeys-random   ← random eviction (rarely ideal)
    noeviction       ← reject new writes when full (default)
```

---

### 📐 LRU vs LFU — The Interview Showdown

```
Scenario: Cache has 3 slots
Access pattern: A,A,A,A,A,B,B,B,C,C → new item D arrives

LRU view:
  Most recently accessed order: C(last), B, A
  Evicts: A (oldest access)
  Problem: A was accessed 5 times! It's popular but not recent. ❌

LFU view:
  Frequency counts: A=5, B=3, C=2
  Evicts: C (lowest frequency count)
  Result: The truly least popular item is evicted ✅

Verdict:
  LRU → recency wins (good for time-windowed traffic)
  LFU → popularity wins (good for stable hot-set workloads)
```

---

## 7. Quick Revision Cheatsheet

```
CACHING FUNDAMENTALS
────────────────────────────────────────────────────────────────
Cache     = Fast temp storage. Memory over disk. <1ms vs 100ms.
Cache Hit = Data found in cache → serve instantly ✅
Cache Miss= Data not in cache → fetch DB → store in cache → serve ❌→✅
Hit Rate  = Hits/(Hits+Misses) × 100. Target: >90% in production.
TTL       = Expiry timer on cached data. Prevents stale data + saves memory.

TTL GUIDELINES
────────────────────────────────────────────────────────────────
Stock prices / live rates   → 1–10 seconds
User sessions / tokens      → 30–60 minutes
User profiles               → 1–6 hours
Product catalog / listings  → 6–24 hours
Static assets / configs     → 1–7 days

WRITE STRATEGIES
────────────────────────────────────────────────────────────────
Read Through   = App → Cache → (miss?) Cache fetches DB itself
Write Through  = Write to Cache + DB simultaneously. Always consistent. Slower writes.
Write Around   = Write to DB only. Cache populated on read miss. Good for write-heavy data.
Write Back     = Write to Cache first, DB async later. Fastest. Risk: data loss on crash.

EVICTION POLICIES
────────────────────────────────────────────────────────────────
LRU   = Evict Least Recently Used.    Best for general caching. Redis default.
LFU   = Evict Least Frequently Used.  Best for popularity-based. Keeps viral content.
FIFO  = Evict oldest inserted.        Simple queues and buffers.
MRU   = Evict Most Recently Used.     Sequential scans. Opposite of LRU.
LIFO  = Evict most recently inserted. Undo stacks. Algorithm backtracking.

GOLDEN RULES
────────────────────────────────────────────────────────────────
→ Cache is a read optimization — mostly helps reads, not writes
→ Every cache needs a TTL — stale data is worse than slow data
→ Cache != Source of truth — DB is always the source of truth
→ Write Back is fastest but loses data on crash — never for financial data
→ LRU is the safe default. LFU when popularity matters more than recency.
→ Monitor hit rate — below 80% means your cache strategy needs rethinking
```

---

*Made for interview prep — understand the concept, remember the line, nail the answer.*